# 00 — Technical Audit of the DeepHS Fruit v2 Persimmon Subset

This notebook performs the definitive pre-modeling audit used by the manuscript. It reads only the three physical v2 annotation files (`train_all_v2.json`, `val_v2.json`, `test_v2.json`), explicitly excludes `train_only_labeled_v2.json` because it is a labeled subset of `train_all_v2.json`, verifies the temporal target mapping, validates the HSI file inventory, and writes the master index used by all downstream notebooks.

No predictive model is trained here.


### Public reproducibility version

Paths are repository-relative. Raw DeepHS Fruit data are not redistributed. Download `Kaki.zip` and `annotations-upd-2024-01-09.zip` with `python scripts/download_data.py` before running this notebook.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Imports and paths


In [ ]:
from datetime import datetime
from collections import defaultdict
import hashlib
import json
import re
import shutil
import sys
import platform
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "00_kaki_data_audit_corrected_v2"

ROOT = PROJECT_ROOT
RAW = ROOT / "data/raw"
INTERIM = ROOT / "data/interim"
ANNOT_DIR = INTERIM / "annotations_extracted"
EXTRACTED = INTERIM / "extracted"

TABLES = ROOT / "tables/00_kaki_data_audit"
RESULTS = ROOT / "results/00_kaki_data_audit"
FIGURES = ROOT / "figures/00_kaki_data_audit"

for p in [INTERIM, ANNOT_DIR, EXTRACTED, TABLES, RESULTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

ANNOT_ZIP = RAW / "annotations-upd-2024-01-09.zip"
KAKI_ZIP = RAW / "Kaki.zip"

EXPECTED_HASHES = {
    "annotations": "14275450e362684bc379a5aaf6c845cf82b0f9d5912036b9401d59a4b964a3f3",
    "kaki": "c388a528391e975c815f79859ef98f8966ecc06ec2426ec2d5bc6880efb717f6",
}

OFFICIAL_V2_JSONS = {
    "train_all_v2": "annotations/train_all_v2.json",
    "val_v2": "annotations/val_v2.json",
    "test_v2": "annotations/test_v2.json",
}

DAY_TO_STORAGE = {
    "day_1_m3": 0,
    "day_2_m3": 1,
    "day_3_m3": 2,
    "day_4_m3": 3,
    "day_5_m3": 4,
    "day_7_m3": 6,
    "day_8_m3": 7,
    "day_9_m3": 8,
}

EXPECTED_DAYS = [0, 1, 2, 3, 4, 6, 7, 8]
print("Raw:", RAW)


## 2. Verify raw archives and SHA-256


In [ ]:
def sha256_file(path, block_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

for p in [ANNOT_ZIP, KAKI_ZIP]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing {p}. Run: python scripts/download_data.py"
        )
    if not zipfile.is_zipfile(p):
        raise RuntimeError(f"Not a valid ZIP: {p}")

annot_hash = sha256_file(ANNOT_ZIP)
kaki_hash = sha256_file(KAKI_ZIP)

assert annot_hash == EXPECTED_HASHES["annotations"], annot_hash
assert kaki_hash == EXPECTED_HASHES["kaki"], kaki_hash

raw_manifest = {
    "annotations_zip": {
        "path": str(ANNOT_ZIP),
        "size_bytes": ANNOT_ZIP.stat().st_size,
        "sha256": annot_hash,
    },
    "kaki_zip": {
        "path": str(KAKI_ZIP),
        "size_bytes": KAKI_ZIP.stat().st_size,
        "sha256": kaki_hash,
    },
}
display(pd.DataFrame(raw_manifest).T)


## 3. Extract annotations and select only physical v2 partitions

`train_only_labeled_v2.json` is **not** appended. It duplicates the labeled subset already present in `train_all_v2.json`.


In [ ]:
if ANNOT_DIR.exists():
    shutil.rmtree(ANNOT_DIR)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ANNOT_ZIP, "r") as z:
    z.extractall(ANNOT_DIR)

all_json_names = sorted(
    str(p.relative_to(ANNOT_DIR)).replace("\\", "/")
    for p in ANNOT_DIR.rglob("*.json")
)

required = set(OFFICIAL_V2_JSONS.values())
missing = required - set(all_json_names)
if missing:
    raise RuntimeError(f"Missing official v2 annotation files: {sorted(missing)}")

selected_annotation_files = {
    split: ANNOT_DIR / rel_path
    for split, rel_path in OFFICIAL_V2_JSONS.items()
}

print("Selected:")
for split, path in selected_annotation_files.items():
    print(f"  {split}: {path.relative_to(ANNOT_DIR)}")
print("Explicitly excluded: annotations/train_only_labeled_v2.json")


## 4. Normalize records and annotations


In [ ]:
all_rows = []

for split_name, json_path in selected_annotation_files.items():
    obj = json.loads(json_path.read_text(encoding="utf-8"))
    annotations = {
        int(a["record_id"]): a
        for a in obj.get("annotations", [])
    }

    for r in obj.get("records", []):
        if r.get("fruit") != "Kaki":
            continue

        rid = int(r["id"])
        a = annotations.get(rid, {})
        files = r.get("files", {}) or {}

        row = {
            "official_split_source": split_name,
            "record_id": rid,
            "record_key": f"{split_name}:{rid}",
            "fruit": r.get("fruit"),
            "side": r.get("side"),
            "day_code": r.get("day"),
            "camera_type": r.get("camera_type"),
            "header_file": files.get("header_file"),
            "data_file": files.get("data_file"),
        }

        for k, v in a.items():
            if k not in {"id", "record_id"}:
                row[k] = v

        all_rows.append(row)

kaki = pd.DataFrame(all_rows)

assert len(kaki) == 793, len(kaki)
assert kaki["record_key"].is_unique
assert set(kaki["official_split_source"]) == {
    "train_all_v2", "val_v2", "test_v2"
}

split_counts = (
    kaki.groupby("official_split_source")
    .size()
    .rename("n_cubes")
    .reset_index()
)
display(split_counts)
assert dict(zip(split_counts["official_split_source"], split_counts["n_cubes"])) == {
    "train_all_v2": 745,
    "val_v2": 24,
    "test_v2": 24,
}


## 5. Reconstruct persistent physical fruit IDs


In [ ]:
FRUIT_RE = re.compile(
    r"kaki_day_\d+_m3_(\d+)_",
    flags=re.IGNORECASE,
)

def parse_fruit_id(path):
    if not isinstance(path, str):
        return np.nan
    m = FRUIT_RE.search(path.replace("\\", "/"))
    return int(m.group(1)) if m else np.nan

kaki["fruit_id"] = kaki["header_file"].map(parse_fruit_id)

if kaki["fruit_id"].isna().any():
    raise RuntimeError(
        f"Could not reconstruct fruit_id for {int(kaki['fruit_id'].isna().sum())} records."
    )

kaki["fruit_id"] = kaki["fruit_id"].astype(int)

assert kaki["fruit_id"].nunique() == 40
assert kaki["fruit_id"].min() == 1
assert kaki["fruit_id"].max() == 40

print("Physical fruits:", kaki["fruit_id"].nunique())


## 6. Verify and propagate the storage-day target


In [ ]:
kaki["storage_days_original"] = pd.to_numeric(
    kaki.get("storage_days"),
    errors="coerce",
)

official_pairs = (
    kaki.loc[
        kaki["storage_days_original"].notna(),
        ["day_code", "storage_days_original"]
    ]
    .drop_duplicates()
    .sort_values("storage_days_original")
)

expected_pairs = pd.DataFrame({
    "day_code": list(DAY_TO_STORAGE.keys()),
    "storage_days_original": list(DAY_TO_STORAGE.values()),
})

merged_pairs = expected_pairs.merge(
    official_pairs,
    on=["day_code", "storage_days_original"],
    how="outer",
    indicator=True,
)

if not (merged_pairs["_merge"] == "both").all():
    raise RuntimeError(
        "Official destructive annotations do not match the verified day-code mapping."
    )

kaki["storage_days_model"] = kaki["day_code"].map(DAY_TO_STORAGE)

if kaki["storage_days_model"].isna().any():
    unknown = sorted(kaki.loc[kaki["storage_days_model"].isna(), "day_code"].unique())
    raise RuntimeError(f"Unknown Kaki day codes: {unknown}")

kaki["storage_days_model"] = kaki["storage_days_model"].astype(int)

kaki["storage_days_source"] = np.where(
    kaki["storage_days_original"].notna(),
    "official_annotation",
    "derived_from_verified_day_code_mapping",
)

assert sorted(kaki["storage_days_model"].unique().tolist()) == EXPECTED_DAYS
assert int(kaki["storage_days_original"].notna().sum()) == 160

mapping_table = pd.DataFrame(
    [{"day_code": k, "storage_days_model": v} for k, v in DAY_TO_STORAGE.items()]
)
mapping_table.to_csv(
    TABLES / "verified_day_to_storage_mapping.csv",
    index=False,
)
display(mapping_table)


## 7. Audit longitudinal structure, cameras, sides, and terminal endpoints


In [ ]:
cube_counts = (
    kaki.groupby("storage_days_model")
    .size()
    .rename("n_cubes")
    .reset_index()
)
fruit_counts = (
    kaki.groupby("storage_days_model")["fruit_id"]
    .nunique()
    .rename("n_fruits")
    .reset_index()
)
camera_counts = (
    kaki.groupby("camera_type")
    .size()
    .rename("n_cubes")
    .reset_index()
)

expected_cube_counts = {0:157, 1:152, 2:132, 3:112, 4:92, 6:76, 7:48, 8:24}
expected_fruit_counts = {0:40, 1:38, 2:33, 3:28, 4:23, 6:19, 7:12, 8:6}
expected_camera_counts = {"VIS":395, "VIS_COR":398}

assert dict(zip(cube_counts["storage_days_model"], cube_counts["n_cubes"])) == expected_cube_counts
assert dict(zip(fruit_counts["storage_days_model"], fruit_counts["n_fruits"])) == expected_fruit_counts
assert dict(zip(camera_counts["camera_type"], camera_counts["n_cubes"])) == expected_camera_counts

fruit_day_structure = (
    kaki.groupby(["fruit_id", "storage_days_model"])
    .agg(
        n_cubes=("record_key", "size"),
        n_cameras=("camera_type", "nunique"),
        n_sides=("side", "nunique"),
        n_official_annotations=("storage_days_original", lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
assert len(fruit_day_structure) == 199
assert int((fruit_day_structure["n_cubes"] == 4).sum()) == 196
assert int((fruit_day_structure["n_cubes"] == 3).sum()) == 3

terminal_observed = (
    kaki.groupby("fruit_id")["storage_days_model"]
    .max()
    .rename("terminal_observed_day")
)
terminal_labeled = (
    kaki.loc[kaki["storage_days_original"].notna()]
    .groupby("fruit_id")["storage_days_model"]
    .unique()
)

assert len(terminal_labeled) == 40
for fruit_id, arr in terminal_labeled.items():
    assert len(arr) == 1
    assert int(arr[0]) == int(terminal_observed.loc[fruit_id])

ordered_days = EXPECTED_DAYS
sequence_rows = []
for fruit_id, g in kaki.groupby("fruit_id"):
    observed = sorted(g["storage_days_model"].unique().tolist())
    max_day = max(observed)
    expected_prefix = ordered_days[:ordered_days.index(max_day)+1]
    is_complete_prefix = observed == expected_prefix
    sequence_rows.append({
        "fruit_id": int(fruit_id),
        "observed_days": "|".join(map(str, observed)),
        "terminal_day": int(max_day),
        "complete_prefix": bool(is_complete_prefix),
    })
    assert is_complete_prefix

sequence_check = pd.DataFrame(sequence_rows)

for frame, name in [
    (cube_counts, "storage_day_cube_counts_v2.csv"),
    (fruit_counts, "storage_day_fruit_counts_v2.csv"),
    (camera_counts, "camera_counts_v2.csv"),
    (fruit_day_structure, "fruit_day_structure_v2.csv"),
    (sequence_check, "longitudinal_sequence_check_v2.csv"),
]:
    frame.to_csv(TABLES / name, index=False)

display(cube_counts)
display(fruit_counts)
display(camera_counts)


## 8. Audit overlap in the official record-level partitions


In [ ]:
fruit_split = (
    kaki.groupby("fruit_id")["official_split_source"]
    .agg(lambda s: sorted(set(s)))
    .rename("official_splits")
    .reset_index()
)
fruit_split["n_official_splits"] = fruit_split["official_splits"].map(len)
fruit_split["official_splits"] = fruit_split["official_splits"].map("|".join)

n_overlap = int((fruit_split["n_official_splits"] > 1).sum())
assert n_overlap == 25, n_overlap

fruit_split.to_csv(
    TABLES / "official_split_overlap_v2.csv",
    index=False,
)

print("Fruits appearing in >1 official split:", n_overlap)
print("Decision: use custom frozen fruit-disjoint partitions for temporal modeling.")


## 9. Extract Kaki.zip without modifying the raw archive


In [ ]:
EXTRACTION_MARKER = EXTRACTED / ".kaki_extraction_complete"

needs_extract = (
    not EXTRACTION_MARKER.exists()
    or not (EXTRACTED / "Kaki").is_dir()
)

if needs_extract:
    if (EXTRACTED / "Kaki").exists():
        shutil.rmtree(EXTRACTED / "Kaki")

    with zipfile.ZipFile(KAKI_ZIP, "r") as z:
        z.extractall(EXTRACTED)

    EXTRACTION_MARKER.write_text(
        datetime.now().isoformat(timespec="seconds"),
        encoding="utf-8",
    )

KAKI_DIR = EXTRACTED / "Kaki"
if not KAKI_DIR.is_dir():
    raise RuntimeError(f"Expected extracted directory not found: {KAKI_DIR}")

print("Extracted Kaki directory:", KAKI_DIR)


## 10. Verify all HDR/BIN pairs and binary sizes


In [ ]:
def parse_envi_header(path):
    text = Path(path).read_text(encoding="latin-1")
    meta = {}

    def find_scalar(key, cast=str, default=None):
        m = re.search(
            rf"(?im)^\s*{re.escape(key)}\s*=\s*([^\r\n]+)",
            text,
        )
        if not m:
            return default
        value = m.group(1).strip().strip("{}")
        return cast(value)

    meta["samples"] = find_scalar("samples", int)
    meta["lines"] = find_scalar("lines", int)
    meta["bands"] = find_scalar("bands", int)
    meta["header_offset"] = find_scalar("header offset", int, 0)
    meta["data_type"] = find_scalar("data type", int)
    meta["interleave"] = find_scalar("interleave", str)
    meta["byte_order"] = find_scalar("byte order", int, 0)
    return meta

ENVI_DTYPE_BYTES = {
    1: 1,   # uint8
    2: 2,   # int16
    3: 4,   # int32
    4: 4,   # float32
    5: 8,   # float64
    12: 2,  # uint16
    13: 4,  # uint32
    14: 8,  # int64
    15: 8,  # uint64
}

physical_files = [
    p for p in KAKI_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".hdr", ".bin"}
]
assert len(physical_files) == 1586, len(physical_files)

hdr_files = sorted(p for p in physical_files if p.suffix.lower() == ".hdr")
bin_files = sorted(p for p in physical_files if p.suffix.lower() == ".bin")
assert len(hdr_files) == 793
assert len(bin_files) == 793

pair_rows = []
for hdr in hdr_files:
    bin_path = hdr.with_suffix(".bin")
    if not bin_path.exists():
        raise RuntimeError(f"Missing BIN pair for {hdr}")

    meta = parse_envi_header(hdr)
    dtype_bytes = ENVI_DTYPE_BYTES.get(meta["data_type"])
    if dtype_bytes is None:
        raise RuntimeError(f"Unsupported ENVI data type {meta['data_type']}: {hdr}")

    expected_bytes = (
        int(meta["header_offset"])
        + int(meta["samples"])
        * int(meta["lines"])
        * int(meta["bands"])
        * dtype_bytes
    )
    actual_bytes = bin_path.stat().st_size

    pair_rows.append({
        "relative_hdr": str(hdr.relative_to(EXTRACTED)).replace("\\", "/"),
        "relative_bin": str(bin_path.relative_to(EXTRACTED)).replace("\\", "/"),
        "samples": meta["samples"],
        "lines": meta["lines"],
        "bands": meta["bands"],
        "data_type": meta["data_type"],
        "interleave": str(meta["interleave"]).lower(),
        "expected_bin_bytes": expected_bytes,
        "actual_bin_bytes": actual_bytes,
        "size_match": bool(expected_bytes == actual_bytes),
    })

pair_df = pd.DataFrame(pair_rows)

assert pair_df["size_match"].all()
assert set(pair_df["samples"].unique()) == {64}
assert set(pair_df["lines"].unique()) == {64}
assert set(pair_df["data_type"].unique()) == {4}
assert set(pair_df["interleave"].unique()) == {"bip"}

pair_df.to_csv(
    TABLES / "physical_hdr_bin_audit_v2.csv",
    index=False,
)

display(
    pair_df.groupby("bands")
    .size()
    .rename("n_cubes")
    .reset_index()
)


## 11. Cross-reference annotations against physical files and write the master index


In [ ]:
kaki["header_abs"] = kaki["header_file"].map(
    lambda x: str((EXTRACTED / x).resolve())
)
kaki["data_abs"] = kaki["data_file"].map(
    lambda x: str((EXTRACTED / x).resolve())
)

kaki["header_exists"] = kaki["header_abs"].map(
    lambda x: Path(x).is_file()
)
kaki["data_exists"] = kaki["data_abs"].map(
    lambda x: Path(x).is_file()
)
kaki["both_files_exist"] = (
    kaki["header_exists"] & kaki["data_exists"]
)

assert kaki["both_files_exist"].all()

physical_rel = {
    str(p.relative_to(EXTRACTED)).replace("\\", "/")
    for p in physical_files
}
annotated_rel = set(kaki["header_file"]) | set(kaki["data_file"])
assert physical_rel == annotated_rel

preferred_columns = [
    "official_split_source",
    "record_id",
    "record_key",
    "fruit",
    "fruit_id",
    "day_code",
    "storage_days_original",
    "storage_days_model",
    "storage_days_source",
    "camera_type",
    "side",
    "header_file",
    "data_file",
    "header_abs",
    "data_abs",
    "header_exists",
    "data_exists",
    "both_files_exist",
    "init_weight",
    "end_weight",
    "firmness",
    "ripeness_state",
    "ripeness_state_fine",
    "comment",
]
preferred_columns = [c for c in preferred_columns if c in kaki.columns]
remaining = [c for c in kaki.columns if c not in preferred_columns]
master = kaki[preferred_columns + remaining].copy()

MASTER_PATH = INTERIM / "kaki_master_index_v2.csv"
master.to_csv(MASTER_PATH, index=False)

assert len(master) == 793
assert master["fruit_id"].nunique() == 40
assert master[["fruit_id", "storage_days_model"]].drop_duplicates().shape[0] == 199

print("Master index:", MASTER_PATH)
print("Rows:", len(master))


## 12. Final audit summary


In [ ]:
endpoint_counts = (
    terminal_observed.value_counts()
    .sort_index()
    .rename_axis("terminal_day")
    .reset_index(name="n_fruits")
)

expected_endpoint_counts = {0:2, 1:5, 2:5, 3:5, 4:4, 6:7, 7:6, 8:6}
assert dict(zip(endpoint_counts["terminal_day"], endpoint_counts["n_fruits"])) == expected_endpoint_counts

summary = {
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK_ID,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "annotations_sha256": annot_hash,
    "kaki_zip_sha256": kaki_hash,
    "physical_hsi_cubes": 793,
    "physical_fruits": 40,
    "fruit_days": 199,
    "officially_labeled_cubes": 160,
    "terminal_labeled_fruit_days": 40,
    "camera_counts": expected_camera_counts,
    "storage_days": EXPECTED_DAYS,
    "cube_counts_by_day": expected_cube_counts,
    "fruit_counts_by_day": expected_fruit_counts,
    "official_split_fruit_overlap_count": n_overlap,
    "all_hdr_bin_pairs_complete": True,
    "all_bin_sizes_match": True,
    "master_index": str(MASTER_PATH.relative_to(ROOT)),
    "critical_rule": (
        "train_only_labeled_v2 is excluded because it is a labeled subset "
        "of train_all_v2 and must not be appended as an independent partition."
    ),
}

SUMMARY_PATH = RESULTS / "kaki_data_audit_summary_v2.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

endpoint_counts.to_csv(
    TABLES / "destructive_endpoint_counts_v2.csv",
    index=False,
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


## 13. Compact diagnostic figures


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.bar(
    cube_counts["storage_days_model"].astype(str),
    cube_counts["n_cubes"],
)
ax.set_xlabel("Storage day")
ax.set_ylabel("HSI cubes")
ax.set_title("Persimmon HSI cube coverage by storage day")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(
    FIGURES / "01_storage_days_cube_distribution_v2.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

presence = (
    master[["fruit_id", "storage_days_model"]]
    .drop_duplicates()
    .assign(present=1)
    .pivot(index="fruit_id", columns="storage_days_model", values="present")
    .fillna(0)
    .reindex(columns=EXPECTED_DAYS)
)

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(presence.to_numpy(), aspect="auto", interpolation="nearest")
ax.set_xticks(range(len(EXPECTED_DAYS)))
ax.set_xticklabels(EXPECTED_DAYS)
ax.set_xlabel("Storage day")
ax.set_ylabel("Fruit ID")
ax.set_title("Longitudinal fruit × storage-day coverage")
fig.tight_layout()
fig.savefig(
    FIGURES / "02_fruit_storage_day_coverage_v2.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


## Audit decision

If all assertions pass, the dataset is approved for downstream analysis using the persistent physical fruit ID as the biological grouping unit. The official DeepHS record-level train/validation/test split is retained only as provenance and is not used as the temporal model split.
